# 2.10 Python

This chapter's simulation work centres on two ideas: verifying the frequentist interpretation of conditional probability and exploring the Monty Hall problem. Both lend themselves naturally to large-scale simulation.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(palette="deep")
rng = np.random.default_rng(seed=42)

## Simulating the Frequentist Interpretation

The frequentist interpretation of conditional probability says that, over a large number $n$ of repetitions of an experiment,

$$P(A \mid B) \approx \frac{n_{AB}}{n_B},$$

where $n_{AB}$ is the number of trials in which both $A$ and $B$ occurred and $n_B$ is the number in which $B$ occurred. We can verify this using the two-children scenario from Example 2.2.5.

We simulate $n$ families, each with two children. Each child is independently assigned gender 1 (girl) or 2 (boy) with equal probability:

In [ ]:
n = 10**5
child1 = rng.integers(1, 3, size=n)   # elder child: 1=girl, 2=boy
child2 = rng.integers(1, 3, size=n)   # younger child

`child1` and `child2` are arrays of length $n$, each element being 1 or 2. Working with numeric codes is more convenient for counting and logical operations than using strings, though the string version is equally valid:

In [ ]:
# Equivalent string version — less convenient for arithmetic
rng.choice(["girl", "boy"], size=8)

### P(both girls | elder is a girl)

Let $A$ be the event that both children are girls and $B$ the event that the elder child is a girl. We count the trials in which $B$ occurred (`n_b`) and those in which both $A$ and $B$ occurred (`n_ab`), then divide:

In [ ]:
n_b  = np.sum(child1 == 1)                      # elder is a girl
n_ab = np.sum((child1 == 1) & (child2 == 1))    # both are girls
n_ab / n_b

The `&` operator performs element-wise logical AND on Boolean arrays, so `(child1 == 1) & (child2 == 1)` is `True` only for families where both children are girls. The result is approximately 0.50, consistent with the theoretical value $P(\text{both girls} \mid \text{elder is a girl}) = \tfrac{1}{2}$.

### P(both girls | at least one girl)

Now let $B$ be the event that *at least one* child is a girl. The event $A \cap B$ is unchanged (both girls implies at least one girl), but $n_B$ now counts families where either child — or both — is a girl:

In [ ]:
n_b  = np.sum((child1 == 1) | (child2 == 1))    # at least one girl
n_ab = np.sum((child1 == 1) & (child2 == 1))    # both girls
n_ab / n_b

The `|` operator is element-wise logical OR — it returns `True` when at least one of the two conditions holds. The simulated probability comes out around 0.33, matching the theoretical $P(\text{both girls} \mid \text{at least one girl}) = \tfrac{1}{3}$.

Notice how drastically the conditioning event changes the answer: the probability of both children being girls drops from $\tfrac{1}{2}$ to $\tfrac{1}{3}$ depending on whether we know the *elder* child is a girl or merely that *some* child is.

## Monty Hall Simulation

The Monty Hall problem has prompted lengthy arguments that a direct simulation quickly settles. The setup: a car is hidden behind one of three doors, the contestant picks a door, and then the host (who knows where the car is) opens a different door that does not reveal the car. The contestant may then stick with their original choice or switch to the remaining closed door.

### Never-switch strategy

We simulate $10^5$ rounds, assuming the contestant always picks door 1. The car's location is drawn uniformly from {1, 2, 3}:

In [ ]:
n = 10**5
cardoor = rng.integers(1, 4, size=n)   # which door hides the car (1, 2, or 3)

np.sum(cardoor == 1) / n               # never-switch win rate

The never-switch strategy wins exactly when door 1 already has the car — no need to simulate Monty's move at all. The result is close to $\tfrac{1}{3}$.

### Always-switch strategy

After Monty opens a losing door, switching wins whenever the car is *not* behind door 1 — the contestant's original pick. That covers the remaining two doors:

In [ ]:
np.sum(cardoor != 1) / n               # always-switch win rate

This comes out near $\tfrac{2}{3}$, confirming that switching doubles the probability of winning. The two results are complementary — every round is a win for exactly one of the two strategies.

### Visualising the strategies

A bar chart makes the gap between the two strategies immediately visible:

In [ ]:
win_rates = {
    "Never switch": np.sum(cardoor == 1) / n,
    "Always switch": np.sum(cardoor != 1) / n,
}

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=list(win_rates.keys()), y=list(win_rates.values()), ax=ax)
ax.axhline(1/3, color="grey", linestyle="--", linewidth=0.8)
ax.axhline(2/3, color="grey", linestyle="--", linewidth=0.8)
ax.set_ylabel("Win rate")
ax.set_title("Monty Hall: simulated win rates")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Interactive Monty Hall Game

Beyond bulk simulation, we can play through a single round by calling a function. `monty(chosen, switch)` places the car randomly, reveals a losing door that is neither the contestant's pick nor the car door, and then applies the contestant's decision to stay or switch — announcing the outcome at the end.

Pass `chosen` as the door number (1, 2, or 3) and `switch` as `True` or `False`.

In [ ]:
import random

def monty(chosen, switch):
    doors = [1, 2, 3]

    # randomly place the car
    cardoor = random.choice(doors)

    # Monty opens a door that is neither the player's choice nor the car door
    if chosen != cardoor:
        montydoor = [d for d in doors if d != chosen and d != cardoor][0]
    else:
        montydoor = random.choice([d for d in doors if d != chosen])

    print(f"Monty opens door {montydoor}!")

    if switch:
        chosen = [d for d in doors if d != chosen and d != montydoor][0]

    if chosen == cardoor:
        print("You won!")
    else:
        print("You lost!")

Try a few rounds with each strategy to build intuition for why switching is advantageous:

In [ ]:
monty(chosen=1, switch=True)   # pick door 1, always switch